In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
from PIL import Image
import buchineri as buchineri
import importlib
importlib.reload(buchineri)
from buchineri import schwarzschild
from buchineri import immagine
from buchineri import kerr
from pathlib import Path

# create output directories
outschw = Path("immagini/schwarzschild")  
outkerr = Path("immagini/kerr")  
outdatakerr= Path("datikerr")   
outgraf = Path("grafici")
insfondi= Path("sfondi")
outschw.mkdir(parents=True, exist_ok=True)
outkerr.mkdir(parents=True, exist_ok=True)
outdatakerr.mkdir(parents=True, exist_ok=True)
outgraf.mkdir(parents=True, exist_ok=True)
insfondi.mkdir(parents=True,exist_ok=True )

# names of the images to be used as background
f1="milkyway16k.png"
f2="griglia_con_stella.png"
f3="Gaia1.png"
f4="Gaia2.png"
f5="sfondo_glimpse_16k.png"

# plotting style ######
TEXTWIDTH_PT = 426.79135 # width of the text column of the latex document in typographic points
def figsize(frac=1.0, ratio=0.62):
    # frac= width of the imagine in units of the page width, ratio= heigh/width
    w = TEXTWIDTH_PT / 72.27 * frac #width of the imagine in inches 
    return (w, w * ratio) #width and higt of the image
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 10,          # come il corpo del testo, o 1 pt in meno
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.constrained_layout.use": True,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
})
######################

Image.MAX_IMAGE_PIXELS = None   # remove the limit on the png files

def generaimmagine(sfondo, phi_pixel, black, r0, th0,FOV, N, tipo, directory, th_pixel=None):
    nomefile=f"{tipo}_{sfondo}_N={N}_r0={r0}_th={np.rad2deg(th0)}°_FOV={np.rad2deg(FOV)}°"
    imgvuoto,imgbuco=immagine(f"sfondi/{sfondo}",phi_pixel ,black,FOV,N, th0,th_pixel)
    Image.fromarray((imgbuco[::-1] * 255 + 0.5).astype(np.uint8)).save(f"{directory}/{nomefile}.png")
    Image.fromarray((imgvuoto[::-1] * 255 + 0.5).astype(np.uint8)).save(f"{directory}/vuoto_{nomefile}.png")

# Schwarzschild

In [ ]:
#       VARIABILI       #
M=1                     # Massa 
r0=30*M                  # distanza dal buco nero
FOV=70*2*np.pi/360      # field of view
N=1000                  # numero di pixel per lato dell'immagine PARI
#########################

## Immagini

In [ ]:
# calculate the trajectories
phi_pixel_sch,black_sch=schwarzschild(M,r0,FOV,N)

In [ ]:
# generate the image
generaimmagine(f2, phi_pixel_sch, black_sch, r0, np.pi/2, FOV, N, "schw", outschw)

## Validazione Schwarzschild

In [ ]:
from buchineri import integratore_schwarzschild

def conversione_polari_schw(phi,u):
    # Converte la soluzione u(phi) in coordinate cartesiane per il plot
    r=1/u
    return r*np.cos(phi), r*np.sin(phi)

r0 = 30 * M
b_values = [1,2,3,4,5,5.125,5.25,5.5,6, 7, 8,9,10,11]

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

norm = Normalize(vmin=min(b_values), vmax=max(b_values))   # mappa b -> [0, 1]
cmap = plt.cm.magma

# figura 1
fig1, ax1 = plt.subplots(figsize=figsize(1.0))
cerchio = plt.Circle((0, 0), 2 * M, color='black', label='Orizzonte degli eventi (2M)')
ax1.plot(r0, 0, marker='o', color='black', markersize=7, linestyle='none',
         zorder=11, label=f'Pinhole ($r_0$ = {r0/M:g} M)')
ax1.add_patch(cerchio)
ax1.set_aspect('equal') 
ax1.set_xlabel('x/M')
ax1.set_ylabel('y/M')

# figura 2
fig2, ax2 = plt.subplots(figsize=figsize(0.8, 0.6))
ax2.set_xlabel('$\\phi$ [rad]')
ax2.set_ylabel('Controllo ($u^2 + du^2 - 2Mu^3=1/b^2$)')
ax2.grid(True)

# integrazione
risultati = []
for b in b_values:
    sol, catturato, _ = integratore_schwarzschild(b, r0,M)
    risultati.append((b, sol, catturato))

b_catt = [b for b, _, c in risultati if c]
b_defl = [b for b, _, c in risultati if not c]

# Plotting
for b, sol, catturato in risultati:
    phi = np.linspace(0,sol.t[-1], 1000)
    u, du = sol.sol(phi)
    mask =u>(1/(r0+1e-6))  
    phi = phi[mask]
    u = u[mask]
    du = du[mask]
    x, y = conversione_polari_schw(phi, u)
    stato = 'catturato' if catturato else 'deflesso'
    ax1.plot(x, y, linewidth=1.4, color=cmap(norm(b)))
    controllo = u**2 + du**2 - 2*M*u**3 - 1/b**2
    ax2.plot(phi, controllo, linewidth=1, color=cmap(norm(b)))

cerchio.set_zorder(10) # black hole over the trajectories
sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = fig1.colorbar(sm, ax=ax1, pad=0.02)      
cbar.set_label(r"$b/M$")
sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = fig2.colorbar(sm, ax=ax2, pad=0.02)       
cbar.set_label(r"$b/M$")
# legenda fuori dal grafico, per non coprire le traiettorie
ax1.legend(loc='upper left', fontsize=8)

plt.show()
fig1.savefig(f"{outgraf}/schwarzschild_variousb.pdf", bbox_inches='tight')
fig2.savefig(f"{outgraf}/controlloqualitàschw.pdf", bbox_inches='tight')

# Kerr

In [ ]:
#       VARIABLES       #
M=1                     # Massa 
a=0.99                  # Spin
r0=30*M                 # distanza dal buco nero
th0=np.pi/6             # angolo inziale
FOV=70*2*np.pi/360      # field of view
N=500                   # numero di pixel per lato dell'immagine (N=1000 -> 152 minuti)
#########################

## Immagini

In [ ]:
import subprocess, os

# single processing
caff = subprocess.Popen(["caffeinate", "-i", "-w", str(os.getpid())])
try:
    phi_pixel_kerr,th_pixel,black_kerr=kerr(M,a,r0,th0,FOV,N)
    pass
finally:
    caff.terminate()
np.savez(outdatakerr/f"kerr_data_{M}_{a}_{r0}_{round(np.rad2deg(th0),6)}°_{round(np.rad2deg(FOV))}°_{N}.npz", phi_pixel_kerr=phi_pixel_kerr, th_pixel=th_pixel, black_kerr=black_kerr)


In [ ]:
data = np.load(outdatakerr/f"kerr_data_{M}_{a}_{r0}_{round(np.rad2deg(th0),6)}°_{round(np.rad2deg(FOV))}°_{N}.npz")
phi_pixel_kerr = data["phi_pixel_kerr"]
th_pixel = data["th_pixel"]
black_kerr = data["black_kerr"]

In [ ]:
generaimmagine(f1, phi_pixel_kerr, black_kerr, r0, th0, FOV, N, "kerr", outkerr,th_pixel=th_pixel)

In [ ]:
generaimmagine(f2, phi_pixel_kerr, black_kerr, r0, th0, FOV, N, "kerr", outkerr,th_pixel=th_pixel)

## Validazione Kerr

In [ ]:
from buchineri import integratore_kerr

def conversione_polari_kerr(phi,r):
    return r*np.cos(phi), r*np.sin(phi)

a=0.99
r0 = 30 * M
th0 = np.pi/2

b_values = [-12,-11,-10,-9,-8,-7,-6,-5,-4,-3,-2,-1,1,2,3,4,5,6,7,8,9,10,11,12]

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

norm = Normalize(vmin=min(b_values), vmax=max(b_values))   # mappa b -> [0, 1]
cmap = plt.cm.viridis

# figura 1
fig1, ax1 = plt.subplots(figsize=figsize(1.0))
cerchio = plt.Circle((0, 0), M+np.sqrt(M**2-a**2), color='black', label='Orizzonte degli eventi ($r_+$)')
ax1.plot(r0, 0, marker='o', color='black', markersize=7, linestyle='none',
         zorder=11, label=f'Pinhole ($r_0$ = {r0/M:g} M)')
ax1.add_patch(cerchio)
ax1.set_aspect('equal') 
ax1.set_xlabel('x/M')
ax1.set_ylabel('y/M')

# figura 2
fig2, ax2 = plt.subplots(figsize=figsize(1.0))
ax2.set_xlabel('$\\eta M$ ')
ax2.set_ylabel('Controllo $(\\dot r^2-R(r))/R(r_0)=0$')
ax2.grid(True)

# integrazione
risultati = []
for b in b_values:
    sol, catturato, _,_ = integratore_kerr(b,0, r0,th0,1,M,a)
    risultati.append((b, sol, catturato))

b_catt = [b for b, _, c in risultati if c]
b_defl = [b for b, _, c in risultati if not c]

from buchineri import R
# Plotting
for b, sol, catturato in risultati:
    tau=sol.t
    r, phi, _, dr, _ = sol.y
    mask =r<(r0+1e-6) 
    r = r[mask]
    dr = dr[mask]
    phi = phi[mask]
    tau = tau[mask]
    x, y = conversione_polari_kerr(phi, r)
    stato = 'catturato' if catturato else 'deflesso'
    ax1.plot(x, y, linewidth=1.4, color=cmap(norm(b)))
    controllo = (dr**2-R(r,b,0,M,a))/R(r0,b,0,M,a)  
    ax2.plot(tau, controllo, linewidth=1, color=cmap(norm(b)))

# horizon over the trajectories
cerchio.set_zorder(10)

sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = fig1.colorbar(sm, ax=ax1, pad=0.02)      
cbar.set_label(r"$b/M$")
cbar = fig2.colorbar(sm, ax=ax2, pad=0.02)      
cbar.set_label(r"$b/M$")
fig1.savefig(f"{outgraf}/kerr_variousb.pdf")
fig2.savefig(f"{outgraf}/controlloqualitàkerr.pdf")

plt.show()

## Bisezione per trovare $b_+$ e $b_-$ sul piano equatoriale

In [ ]:
from buchineri import integratore_kerr
th0=np.pi/2
def catturato_kerr(a,b):
    _, cat, _,_ = integratore_kerr(b,0, r0,th0,1,M,a)
    return cat

def bcriticoKerr (a,plus_or_minus):
    if(plus_or_minus):
        lo, hi = 2.0*M, 7.0*M          
    else:    
        lo,hi=-2.0*M,-7.0*M
    # bisezione per calcolare b_c
    for _ in range(30):               
        mid=0.5*(lo + hi)
        if catturato_kerr(a,mid): 
            lo = mid
        else:        
            hi = mid
    return lo/M
b_plus=[]
b_minus=[]
list_a=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.99]
for a in list_a:
    b_plus.append(bcriticoKerr(a,1))
    b_minus.append(bcriticoKerr(a,0))
fig, ax = plt.subplots(figsize=figsize(1.0))
plt.scatter(list_a,b_plus, label='$b_+$ ')
plt.scatter(list_a,-1*np.array(b_minus), label='$|b_-|$')
plt.plot(list_a,3*np.sqrt(3)*M*np.ones(len(list_a)),'--',label='$3\\sqrt{3}$')
plt.xlabel('Spin $a$')
plt.ylabel('$b_c/M$')
# validazione con schwarzschild:
print("confronto |b| critico prograde e retrograde con a=0 e valore esatto da Schwarzschild:", b_plus[0], -1*b_minus[0], 3*np.sqrt(3)*M)
print("b_critico per a=0.99: ",b_plus[-1],b_minus[-1])
rplusph=2*M*(1+np.cos((2/3)*np.arccos(-a/M)))
rminusph=2*M*(1+np.cos((2/3)*np.arccos(a/M)))
bplusteo=a+np.sqrt(rplusph**3/M)
bminusteo=a-np.sqrt(rminusph**3/M)
print("b_critico per a=0.99 teorico: ",bplusteo,bminusteo)
avec= np.linspace(0,0.99,100)
bplusvec=avec+np.sqrt((2*M*(1+np.cos((2/3)*np.arccos(-avec/M))))**3/M)
bminusvec=avec-np.sqrt((2*M*(1+np.cos((2/3)*np.arccos(avec/M))))**3/M)
plt.plot(avec,bplusvec, label='$b_+$ teorico')
plt.plot(avec,-bminusvec, label='$b_-$ teorico')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, frameon=False)
fig.savefig(f"{outgraf}/bcritico_kerr.pdf")

## Altre immagini per Kerr

### $b=\pm 6$

In [ ]:
from buchineri import integratore_kerr
r0=30*M
th0=np.pi/2
a=0.99
abs_b=6*M
fig1, ax1 = plt.subplots(figsize=figsize(0.48, ratio=1))
plt.xlim(-20, 20)
plt.ylim(-20, 20)
ax1.set_xlabel('x/M')
ax1.set_ylabel('y/M')
fig2, ax2 = plt.subplots(figsize=figsize(0.48, ratio=1))
plt.xlim(-20, 20)
plt.ylim(-20, 20)
ax2.set_xlabel('x/M')
ax2.set_ylabel('y/M')

sol1, _, _,_ = integratore_kerr(abs_b,0, r0,th0,1,M,a)
sol2, _, _,_ = integratore_kerr(-abs_b,0, r0,th0,1,M,a)

def conversione_polari_kerr(phi,r):
    return r*np.cos(phi), r*np.sin(phi)

cerchio = plt.Circle((0, 0), M+np.sqrt(M**2-a**2), color='black', label='Orizzonte degli eventi ($r_+$)')
ax2.add_patch(cerchio)
ax2.set_aspect('equal') 
cerchio = plt.Circle((0, 0), M+np.sqrt(M**2-a**2), color='black', label='Orizzonte degli eventi ($r_+$)')
ax1.add_patch(cerchio)
ax1.set_aspect('equal') 


tau1=sol1.t
r1, phi1, _, dr1, _ = sol1.y
mask =r1<(r0+1e-6) 
r1 = r1[mask]
dr1 = dr1[mask]
phi1 = phi1[mask]
tau1 = tau1[mask]
x1, y1 = conversione_polari_kerr(phi1, r1)
ax1.plot(x1, y1, linewidth=1.4, label=f'traiettoria fotone', color='orange')

tau2=sol2.t
r2, phi2, _, dr2, _ = sol2.y
mask =r2<(r0+1e-6) 
r2 = r2[mask]
dr2 = dr2[mask]
phi2 = phi2[mask]
tau2 = tau2[mask]
x2, y2 = conversione_polari_kerr(phi2, r2)
ax2.plot(x2, y2, linewidth=1.4, label=f'traiettoria fotone', color='orange')

ax1.legend(fontsize=7)
ax2.legend(fontsize=7)

cerchio.set_zorder(10)

fig1.savefig(f"{outgraf}/kerr_{abs_b:g}_prograde.pdf")
fig2.savefig(f"{outgraf}/kerr_{abs_b:g}_retrograde.pdf")


### Fotone zamo in caduta libera

In [ ]:
from buchineri import integratore_kerr
r0=30*M
th0=np.pi/2
a=0.99
b=0
fig, ax = plt.subplots(figsize=figsize(1.0))
plt.xlim(-10, 10)
plt.ylim(-10, 10)
ax.set_xlabel('x/M')
ax.set_ylabel('y/M')
sol, _, _,_ = integratore_kerr(b,0, r0,th0,1,M,a)
tau=sol.t
r, phi, _, _, _ = sol.y
mask =r<(r0+1e-6) 
r = r[mask]
phi = phi[mask]
tau = tau[mask]
x, y = conversione_polari_kerr(phi, r)
ax.plot(x, y, linewidth=1.4, label=f'traiettoria fotone con b = 0 ', color='orange')

cerchio = plt.Circle((0, 0), M+np.sqrt(M**2-a**2), color='black', label='Orizzonte degli eventi ($r_+$)')
ax.add_patch(cerchio)
ax.set_aspect('equal') 
cerchio.set_zorder(8)
ax.legend(fontsize=8)
fig.savefig(f"{outgraf}/zamo.pdf", bbox_inches='tight')